# Gradient Boosting (GBM) Classifier & Regressor

**Gradient Boosting**, or **Gradient Boosting Machines (GBM)**, is one of the most powerful and widely used supervised machine learning algorithms. Like AdaBoost, it belongs to the class of **sequential ensemble methods**. However, instead of adjusting sample weights based on classification errors, Gradient Boosting builds new models to predict the **pseudo-residuals** (negative gradients of the loss function) of the previous ensemble's predictions.

In these detailed notes, we will cover:
1. **Core Concept:** Functional Gradient Descent, stage-wise additive modeling, and pseudo-residuals.
2. **Gradient Boosting for Regression (Step-by-Step):** Loss function minimization, pseudo-residual derivation, optimal leaf values (MSE), and prediction updating.
3. **Gradient Boosting for Classification (Step-by-Step):** Logistic loss function, probability calculation, log-odds initialization, pseudo-residuals, optimal leaf output calculations via Taylor expansion, and probability mapping.
4. **AdaBoost vs. Gradient Boosting:** A comparative study of their mechanics.
5. **Key Hyperparameters & Regularization:** Estimators, learning rate (shrinkage), subsampling (Stochastic Gradient Boosting), and max depth.
6. **Practical Python Demos:**
   - **Demo A:** Step-by-Step Gradient Boosting Regressor implementation from scratch on a non-linear toy dataset.
   - **Demo B:** Visualizing the sequential fitting process (how successive trees fit pseudo-residuals and boundary updates).
   - **Demo C:** Training & tuning `GradientBoostingClassifier` using `GridSearchCV` on a synthetic dataset.
   - **Summary Checklist** for Gradient Boosting.

## 1. Core Concepts & Prerequisites

### Functional Gradient Descent
In standard gradient descent, we optimize parameter vectors (weights $\mathbf{w}$ and biases $b$) to minimize a loss function. In **Functional Gradient Descent**, we optimize the *predictions* (functions $F(X)$) directly.
Instead of updating parameters, we add a new function (weak learner $h_m(X)$) to the current prediction $F_{m-1}(X)$ such that it points in the direction of the negative gradient of the loss:

$$F_m(X) = F_{m-1}(X) + \eta \cdot h_m(X)$$

Here, the weak learner $h_m(X)$ is trained to approximate the negative gradient (direction of steepest descent) of the loss function.

### Pseudo-Residuals
The negative gradient of the loss function with respect to the prediction is called the **pseudo-residual**. For a dataset of $N$ samples and a loss function $L(y_i, F(x_i))$, the pseudo-residual $r_{im}$ at iteration $m$ is defined as:

$$r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F(x) = F_{m-1}(x)}$$

If we use the Mean Squared Error (MSE) loss function for regression:

$$L(y_i, F(x_i)) = \frac{1}{2}(y_i - F(x_i))^2$$

The pseudo-residual is:

$$r_{im} = -\left[-(y_i - F_{m-1}(x_i))\right] = y_i - F_{m-1}(x_i)$$

This is exactly the standard residual (prediction error)! Thus, for MSE, the negative gradient is simply the residual.

## 2. Gradient Boosting for Regression: Step-by-Step

Let $S = \{(x_1, y_1), \dots, (x_N, y_N)\}$ be a training set of $N$ samples, where $y_i \in \mathbb{R}$.

### Step 1: Initialize the Model
Initialize the ensemble prediction with a constant value $F_0(x)$ that minimizes the loss function:

$$F_0(x) = \arg\min_\gamma \sum_{i=1}^{N} L(y_i, \gamma)$$

For Squared Error (MSE) Loss:

$$F_0(x) = \arg\min_\gamma \sum_{i=1}^{N} \frac{1}{2}(y_i - \gamma)^2 = \frac{1}{N}\sum_{i=1}^N y_i = \text{mean}(y)$$

---

### Step 2: Iterative Training (for $m = 1$ to $M$)
For each boosting round $m$:

#### A. Compute Pseudo-Residuals
Compute the pseudo-residuals for all training samples $i = 1, \dots, N$:

$$r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F(x) = F_{m-1}(x)}$$

#### B. Fit a Base Learner
Train a weak learner (usually a regression tree) $h_m(x)$ to predict the pseudo-residuals $r_{im}$ as targets. The tree splits the feature space into terminal regions (leaves) $R_{jm}$ for $j = 1, \dots, J_m$ (where $J_m$ is the number of leaves).

#### C. Compute Leaf Output Values
For each leaf $R_{jm}$, calculate the optimal value $\gamma_{jm}$ that minimizes the loss when added to the existing predictions:

$$\gamma_{jm} = \arg\min_\gamma \sum_{x_i \in R_{jm}} L(y_i, F_{m-1}(x_i) + \gamma)$$

For MSE Loss, this simplifies to the average of the pseudo-residuals in leaf $R_{jm}$:

$$\gamma_{jm} = \frac{1}{|R_{jm}|} \sum_{x_i \in R_{jm}} r_{im}$$

#### D. Update the Ensemble Prediction
Update the predictions with a learning rate (shrinkage parameter) $\eta \in (0, 1]$:

$$F_m(x) = F_{m-1}(x) + \eta \sum_{j=1}^{J_m} \gamma_{jm} \mathbb{I}(x \in R_{jm})$$

---

### Step 3: Final Output
After $M$ rounds, the final model is:

$$F_M(x) = F_0(x) + \eta \sum_{m=1}^{M} h_m(x)$$

## 3. Gradient Boosting for Classification: Step-by-Step

For binary classification, where targets are $y_i \in \{0, 1\}$.

### Step 1: Initialize the Model
Initialize predictions with log-odds:

$$F_0(x) = \arg\min_\gamma \sum_{i=1}^N L(y_i, \gamma)$$

Using Log Loss (cross-entropy), the optimal constant value is:

$$F_0(x) = \ln\left( \frac{\sum y_i}{\sum (1 - y_i)} \right) = \ln\left( \frac{\text{count of 1s}}{\text{count of 0s}} \right)$$

---

### Step 2: Iterative Training (for $m = 1$ to $M$)
For each boosting round $m$:

#### A. Compute Probabilities & Pseudo-Residuals
First, convert the current log-odds prediction $F_{m-1}(x_i)$ into a probability using the sigmoid function:

$$p_{i, m-1} = \frac{1}{1 + e^{-F_{m-1}(x_i)}}$$

Then compute pseudo-residuals:

$$r_{im} = y_i - p_{i, m-1}$$

#### B. Fit a Regression Tree
Train a regression tree to fit the pseudo-residuals $r_{im}$, yielding terminal regions $R_{jm}$.

#### C. Compute Leaf Output Values
Because we are working in log-odds space, the leaf outputs cannot be simple averages. We solve for $\gamma_{jm}$ using a second-order Taylor expansion approximation of the Log Loss function:

$$\gamma_{jm} \approx \frac{\sum_{x_i \in R_{jm}} r_{im}}{\sum_{x_i \in R_{jm}} p_{i, m-1}(1 - p_{i, m-1})}$$

#### D. Update the Ensemble Prediction
Update the log-odds prediction:

$$F_m(x) = F_{m-1}(x) + \eta \sum_{j=1}^{J_m} \gamma_{jm} \mathbb{I}(x \in R_{jm})$$

---

### Step 3: Final Output
To make predictions for a new sample $x$:
1. Calculate the final log-odds: $F_M(x) = F_0(x) + \eta \sum_{m=1}^{M} h_m(x)$
2. Compute probability: $P(y=1|x) = \frac{1}{1 + e^{-F_M(x)}}$
3. Predict class label:
   $$\hat{y} = \begin{cases} 1 & \text{if } P(y=1|x) \ge 0.5 \\ 0 & \text{otherwise} \end{cases}$$

## 4. AdaBoost vs. Gradient Boosting Comparison

| Feature | AdaBoost | Gradient Boosting |
| :--- | :--- | :--- |
| **Error Handling** | Modifies **instance weights**; misclassified samples get higher weights. | Minimizes loss by fitting trees to the **gradients (pseudo-residuals)** of loss. |
| **Loss Function Optimization** | Explicitly minimizes **Exponential Loss**. | General framework; optimizes **any differentiable loss** (MSE, MAE, Log Loss, Huber). |
| **Base Learner Constraint** | Typically very shallow **Decision Stumps** (depth=1). | Shallow-to-medium **Regression Trees** (typically depth 3 to 6). |
| **Prediction Value** | Learner outputs class labels ($+1, -1$); final prediction is weighted sum of labels. | Learner outputs raw continuous values (gradients); final prediction sums these values. |
| **Robustness to Noise** | Highly sensitive to noisy data and outliers due to exponential weight growth. | More robust; can use robust loss functions (like Huber or MAE) and subsampling. |

## 5. Key Hyperparameters & Regularization

*   **`learning_rate` (Shrinkage $\eta$):** Scales the contribution of each tree. Smaller values (e.g., $0.01$ to $0.1$) require more trees (`n_estimators`) but generalize much better.
*   **`n_estimators`:** Number of trees in the ensemble. Overfitting is possible if this is too high, unlike Random Forest.
*   **`subsample`:** The fraction of training samples to randomly sample for building each tree. If $< 1.0$, this results in **Stochastic Gradient Boosting**, which introduces randomness to reduce variance and speed up training.
*   **`max_depth`:** Maximum depth of individual trees. Typically restricted to $3$–$6$ to keep base learners weak and prevent overfitting.
*   **`loss`:** The loss function to minimize.
    *   *Regression:* `'squared_error'` (MSE), `'absolute_error'` (MAE), `'huber'` (combination), `'quantile'`.
    *   *Classification:* `'log_loss'` (binary/multiclass cross-entropy), `'exponential'` (equivalent to AdaBoost).

## 6. Code Setup & Imports

Let's prepare the Python environment and import the required libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report

# Set visualization contexts
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 7. Demo A: Gradient Boosting Regressor From Scratch

We will implement a simple Gradient Boosting Regressor (`SimpleGradientRegressor`) for Mean Squared Error (MSE) loss from scratch. We will use scikit-learn's `DecisionTreeRegressor` as our base weak learner to fit pseudo-residuals.

In [ ]:
class SimpleGradientRegressor:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.f0 = None
        self.trees = []

    def fit(self, X, y):
        # Step 1: Initialize F0 with the mean of target values
        self.f0 = np.mean(y)
        # Current ensemble predictions
        f_m = np.full_like(y, self.f0, dtype=np.float64)

        for m in range(self.n_estimators):
            # Step 2A: Compute pseudo-residuals (residuals for MSE loss)
            residuals = y - f_m

            # Step 2B: Fit a regression tree to the residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=42 + m)
            tree.fit(X, residuals)

            # Step 2C & 2D: Update predictions
            # The predictions of tree are already the mean residuals in each leaf
            f_m += self.learning_rate * tree.predict(X)
            self.trees.append(tree)

    def predict(self, X):
        # Start predictions with F0
        preds = np.full(X.shape[0], self.f0, dtype=np.float64)
        # Add scaled contributions from each tree
        for tree in self.trees:
            preds += self.learning_rate * tree.predict(X)
        return preds

# Create a non-linear toy dataset
np.random.seed(42)
X_toy = np.sort(5 * np.random.rand(80, 1), axis=0)
y_toy = np.sin(X_toy).ravel() + np.random.normal(0, 0.1, X_toy.shape[0])

# Fit our scratch model
scratch_model = SimpleGradientRegressor(n_estimators=100, learning_rate=0.1, max_depth=2)
scratch_model.fit(X_toy, y_toy)
y_pred_scratch = scratch_model.predict(X_toy)

# Fit scikit-learn model
sklearn_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=42)
sklearn_model.fit(X_toy, y_toy)
y_pred_sklearn = sklearn_model.predict(X_toy)

print(f"Scratch Regressor MSE:      {mean_squared_error(y_toy, y_pred_scratch):.6f}")
print(f"Scikit-learn Regressor MSE: {mean_squared_error(y_toy, y_pred_sklearn):.6f}")
print(f"Are predictions identical?   {np.allclose(y_pred_scratch, y_pred_sklearn, atol=1e-5)}")

## 8. Demo B: Visualizing Boundary Updates & Residual Fitting

To build physical intuition, we will plot how predictions evolve as more estimators are added, and inspect how individual trees target pseudo-residuals.

In [ ]:
# Train models with different estimator counts to inspect prediction paths
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

n_estimators_list = [1, 3, 10, 100]
X_plot = np.linspace(0, 5, 500).reshape(-1, 1)

for idx, n_est in enumerate(n_estimators_list):
    model = SimpleGradientRegressor(n_estimators=n_est, learning_rate=0.1, max_depth=2)
    model.fit(X_toy, y_toy)
    preds = model.predict(X_plot)
    
    ax = axes[idx]
    ax.scatter(X_toy, y_toy, color='#3498db', alpha=0.7, label='Training Data')
    ax.plot(X_plot, preds, color='#e74c3c', lw=3, label=f'GBM (M={n_est})')
    ax.set_title(f"Predictions with M = {n_est} Trees", fontsize=12, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.legend()

plt.suptitle("How Gradient Boosting Learns Sequentially", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Let's inspect the target residuals and tree fits for the first 3 iterations
fig, axes = plt.subplots(3, 1, figsize=(12, 15))
f_current = np.full_like(y_toy, np.mean(y_toy))

for m in range(3):
    residuals = y_toy - f_current
    tree = DecisionTreeRegressor(max_depth=2, random_state=42)
    tree.fit(X_toy, residuals)
    
    ax = axes[m]
    ax.scatter(X_toy, residuals, color='purple', alpha=0.7, label=f'Pseudo-Residuals (m={m+1})')
    ax.plot(X_plot, tree.predict(X_plot), color='orange', lw=2.5, linestyle='--', label=f'Tree Fit h_{m+1}(x)')
    ax.set_title(f"Iteration {m+1}: Fitting tree to current residuals", fontsize=12, fontweight='bold')
    ax.set_ylabel('Residuals')
    ax.legend()
    
    f_current += 0.1 * tree.predict(X_toy)

plt.tight_layout()
plt.show()

## 9. Demo C: Classifier Hyperparameter Tuning via GridSearchCV

Now we will transition to classification and optimize Scikit-Learn's `GradientBoostingClassifier` on a synthetic dataset. We will perform grid-search cross-validation to search for optimal estimators, learning rates, subsampling ratios, and depths.

In [ ]:
# 1. Create a synthetic classification dataset
X_clf, y_clf = make_classification(
    n_samples=600, n_features=10, n_informative=7, n_redundant=3, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

# 2. Baseline Model
baseline_gbm = GradientBoostingClassifier(random_state=42)
baseline_gbm.fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, baseline_gbm.predict(X_test))

# 3. Define the parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4],
    'subsample': [0.8, 1.0]
}

print("Running Grid Search CV...")
grid = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_gbm = grid.best_estimator_
tuned_acc = accuracy_score(y_test, best_gbm.predict(X_test))

print("\nGridSearchCV Results:")
print("="*60)
print(f"Baseline Classifier Accuracy: {baseline_acc:.4f}")
print(f"Tuned Classifier Accuracy:    {tuned_acc:.4f}")
print(f"Best Hyperparameters:       {grid.best_params_}")
print("="*60)
print("\nClassification Report (Tuned Model):")
print(classification_report(y_test, best_gbm.predict(X_test)))

# 4. Plot Feature Importances
importances = best_gbm.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Tuned Gradient Boosting Classifier: Feature Importances", fontsize=14, fontweight='bold')
sns.barplot(x=importances[indices], y=[f"Feature {i}" for i in indices], palette='viridis')
plt.xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

## Summary Checklist for Gradient Boosting

1.  **Core Intuition:** Train a sequence of **Regression Trees** to fit the **pseudo-residuals** (negative gradients of the loss function) of the previous ensemble's predictions.
2.  **Loss Optimization:** Generalizes to any differentiable loss function.
    -   *MSE Regression:* Pseudo-residuals are $y_i - F_{m-1}(x_i)$. Leaf values are the average of the residuals in each leaf.
    -   *Log Loss Classification:* Pseudo-residuals are $y_i - p_i$ (where $p_i$ is probability). Leaf values are estimated via Taylor expansion: $\frac{\sum \text{residuals}}{\sum p_i(1-p_i)}$.
3.  **AdaBoost vs. Gradient Boosting:**
    -   AdaBoost updates **weights of data points** based on correctness.
    -   Gradient Boosting computes **residuals** and fits new learners to correct those residuals directly in the target space.
4.  **Regularization (Preventing Overfitting):**
    -   *Shrinkage:* Scaled by `learning_rate` ($\eta$). A smaller learning rate requires more trees (`n_estimators`) but leads to better generalization.
    -   *Stochastic Gradient Boosting:* Setting `subsample` $< 1.0$ (e.g., $0.8$) uses a random subset of samples to build each tree, adding regularization and speed.
    -   *Tree Constraints:* Keeping `max_depth` small (e.g., $3$–$6$) restricts complexity of base learners.
5.  **Prediction Mechanism:**
    -   *Regression:* Sum of base predictions and all tree predictions.
    -   *Classification:* Sum log-odds contributions, apply sigmoid to find probability, threshold at $0.5$.